# Example code of Michaelis-Menten

We will use parameter estimation on the Michaelis-Menten formula as an example to illustrate how parameter estimation can be done in our code.


## Problem Description

We consider a **parameter estimation problem for an enzymatic reaction model based on the Michaelis–Menten equation**, where reaction parameters are inferred from experimental (or synthetic) data.

In this problem,

- the time evolution of the enzymatic reaction is described by an ordinary differential equation (ODE),
- observational data are obtained under multiple initial substrate concentration conditions,
- and the model parameters are statistically estimated from these observations.

The goal is to determine a set of parameters that consistently explains the time-series data across all experimental conditions.

---

### What is the Michaelis–Menten Equation?

The **Michaelis–Menten equation** is one of the most fundamental rate equations used to describe how the reaction rate of an enzyme depends on the substrate concentration.

In enzyme kinetics:

- When the substrate concentration is low, the reaction rate increases approximately proportionally with substrate concentration.
- When the substrate concentration becomes sufficiently high, the enzyme becomes saturated, and further increases in substrate concentration produce little change in the reaction rate.

This characteristic behavior — **linear increase at low concentration and saturation at high concentration** — is compactly described by the Michaelis–Menten equation.

---

### Mathematical Expression

The Michaelis–Menten equation is expressed as

$
v = \dfrac{V_{\max}[S]}{K_m + [S]}
$

where

- $ v $ : reaction rate  
- $[S]$ : substrate concentration  
- $ V_{\max}$ : maximum reaction rate  
- $ K_m $: Michaelis constant  

---

### Extension to a Time Evolution Model

In this study, the reaction rate expression is used to describe the **time evolution of the substrate concentration** through the following ordinary differential equation:

$
\dfrac{dS(t)}{dt} = - \dfrac{V_{\max} S(t)}{K_m + S(t)}
$

where $ S(t)$ denotes the substrate concentration at time$ t$.

The product concentration $ P(t) $ is calculated from the initial substrate concentration $ S_0 $ as

$
P(t) = S_0 - S(t)
$

---

## Parameters to Estimate

The parameters to be estimated in this problem are as follows.

### Kinetic Parameters

- **Maximum reaction rate $V_{\max}$**  
  Represents the reaction rate when the enzyme is fully saturated with substrate.

- **Michaelis constant $K_m$**  
  The substrate concentration at which the reaction rate equals $V_{\max}/2$.  
  It is often interpreted as an indicator of the affinity between enzyme and substrate.

### Observation Noise Parameter (optional)

- **Observation noise standard deviation $ \sigma$**  
  Represents the variability of observed data around the model prediction.

Depending on the analysis setting,

- $ \sigma $ may be **treated as known and fixed**, or  
- $ \sigma $ may be **estimated simultaneously with the kinetic parameters**.

---

## Generation of Synthetic Data

Instead of using experimental data, **synthetic observational data** are generated using known true parameters.

### Procedure for Generating Synthetic Data

1. Specify the true parameters  

$
V_{\max}^{\mathrm{true}}, \quad K_m^{\mathrm{true}}
$

2. Prepare multiple initial substrate concentrations  

$
S_0^{(1)}, S_0^{(2)}, \dots, S_0^{(N)}
$

3. For each condition, numerically solve the Michaelis–Menten ODE to compute the theoretical product concentration  

$
P_{\mathrm{true}}(t)
$

4. Add Gaussian observation noise at each time point

$
P_{\mathrm{obs}}(t) = P_{\mathrm{true}}(t) + \varepsilon(t)
$

$
\varepsilon(t) \sim \mathcal{N}(0,\sigma^2)
$

This procedure produces synthetic data that mimic realistic experimental observations.

---

### Purpose of Using Synthetic Data

Synthetic data are used for the following purposes:

- validation of the parameter estimation algorithm  
- examination of **parameter identifiability**  
- evaluation of estimation accuracy and uncertainty  

---

### Summary

This problem can be formulated as

> a statistical parameter estimation problem in which the kinetic parameters of a Michaelis–Menten enzymatic reaction model are inferred from time-series data obtained under multiple experimental conditions.

This framework is widely used as a **benchmark problem for evaluating Bayesian inference methods**, including

- Markov Chain Monte Carlo (MCMC)
- Sequential Monte Carlo (SMC)
- other probabilistic inference techniques.

## Parameter Estimation

In this section, we describe the parameter estimation setup for the Michaelis–Menten model and the outputs obtained from the estimation procedure.

---

### Estimation Setup

In this problem, the following parameters are estimated:

- Maximum reaction rate $V_{\max}$
- Michaelis constant $K_m$
- (optionally) observation noise standard deviation $\sigma$

The time evolution of the substrate concentration is modeled by the following ordinary differential equation:

$
\dfrac{dS(t)}{dt} = - \dfrac{V_{\max} S(t)}{K_m + S(t)}
$

The product concentration is defined as

$
P(t) = S_0 - S(t)
$

The observational data are assumed to follow a Gaussian noise model:

$
P_{\mathrm{obs}}(t) = P_{\mathrm{model}}(t) + \varepsilon(t)
$

$
\varepsilon(t) \sim \mathcal{N}(0,\sigma^2)
$

Time-series data obtained under multiple initial substrate concentration conditions \(S_0^{(i)}\) are used simultaneously.  
The parameters $(V_{\max}, K_m, \sigma)$ are assumed to be **shared across all experimental conditions**.

---

### Likelihood Function

The estimation is performed within a Bayesian framework.  
Under the Gaussian observation model, the log-likelihood function is given by

$$
\log p(\mathcal{D} \mid \theta)
=
\sum_{i=1}^{N_{\mathrm{cond}}}
\sum_{j=1}^{N_t}
\left[
-\frac{1}{2}\log(2\pi\sigma^2)
-
\frac{
\left(
P_{\mathrm{obs}}^{(i)}(t_j) - P_{\mathrm{model}}^{(i)}(t_j)
\right)^2
}{2\sigma^2}
\right]
$$

where

$
\theta = \{V_{\max}, K_m, \sigma\}
$

denotes the set of parameters to be estimated.

---

### Implementation

The likelihood function is implemented through a simulation function that solves the ODE model and compares the predicted product concentration with the observed data.

The following function is defined and called from the main program.

In [ ]:
import ray


# --- ODE and simulation ---
def mm_ode(t, S, Vmax, Km):
    return - Vmax * S / (Km + S)

def simulate_mm_on_grid(Vmax, Km, S0, t_array):
    """
    Solve the Michaelis–Menten progress curve on the observation time grid t_array.
    Returns: P_model(t_array)
    """
    t_span = (t_array[0], t_array[-1])

    sol = solve_ivp(
        fun=lambda t, S: mm_ode(t, S, Vmax, Km),
        t_span=t_span,
        y0=[S0],
        t_eval=t_array,
        method="RK45",
    )
    S_model = sol.y[0]
    P_model = S0 - S_model
    return P_model

@ray.remote
def log_likelihood_mm_multi(params):
    """
    Log-likelihood function using data from multiple initial substrate concentration conditions.

    params  : [Vmax, Km, sigma]
    dataset : list of data loaded by load_all_mm_data()
    """

    Vmax, Km, sigma = params

    # If sigma is not estimated, it is fixed to sigma_true
    if est_sigma:
        sigma = params[-1]
    else:
        sigma = sigma_true

    if sigma <= 0:
        return -np.inf

    logL_total = 0.0
    P_model_list = []

    for i in range(n_ex):
        t     = dataset[i]["t"]
        P_obs = dataset[i]["P_obs"]
        S0    = dataset[i]["S0"]

        # Model prediction
        P_model = simulate_mm_on_grid(Vmax, Km, S0, t)

        # Residual
        residual = P_obs - P_model

        logL_i = -0.5 * datapoint * np.log(2*np.pi*sigma**2) \
                 - np.sum(residual**2) / (2*sigma**2)

        logL_total += logL_i
        P_model_list.append(P_model)

    return logL_total, P_model_list

def sim_particle(particle):
    print('sim_particle')

    # Run the likelihood calculation in parallel
    results = [log_likelihood_mm_multi.remote(particle[i, :]) for i in range(n_particle)]

    # Retrieve the parallel computation results
    llk_Cl = ray.get(results)

    # Reshape the returned results
    llk, C_l_ = zip(*llk_Cl)

    return llk, C_l_

### Outputs

The estimation procedure produces the following quantities.

- Estimated values of the parameters $V_{\max}, K_m, (\sigma)$ 
- Posterior distributions (or their approximations) of the parameters  
- Model predictions $P_{\mathrm{model}}(t)$ corresponding to each particle or sample  

These outputs allow us to

- compare the estimated parameters with the true values (in the case of synthetic data),  
- evaluate the uncertainty associated with the parameter estimates,  
- assess the consistency between model predictions and observed data.  

In particular, the ability of a **single parameter set** to consistently explain time-series data obtained under **different initial substrate concentration conditions** is an important indicator for evaluating the validity of the model.